# Задание 5: Сравнение современных моделей генерации музыки

**Цель.** Запустить 5+ современных моделей генерации музыки (с 2022 года) и сгенерировать одну и ту же поп-композицию длиной ~10 секунд каждой из них. Для каждой архитектуры приведено короткое описание принципа работы.

**Выбранные модели (6 штук, чтобы был запас в 2 балла):**

| # | Модель | Год | Семейство архитектуры | Библиотека |
|---|--------|-----|------------------------|------------|
| 1 | MusicGen (Meta) | 2023 | Авторегрессионный трансформер над токенами EnCodec | `transformers` |
| 2 | AudioLDM 2 — music (CVSSP) | 2023 | Latent diffusion с CLAP+T5+GPT-2 кондишенингом | `diffusers` |
| 3 | Riffusion | 2022 | Stable Diffusion 1.5, дообученная на мел-спектрограммах | `diffusers` |
| 4 | MAGNeT (Meta) | 2024 | Неавторегрессионная masked-LM над EnCodec-токенами | `audiocraft` |
| 5 | Bark (Suno) | 2023 | Многоступенчатый трансформер (semantic → coarse → fine) | `transformers` |
| 6 | Stable Audio Open (Stability AI) | 2024 | Diffusion Transformer (DiT) в непрерывном VAE-латенте | `diffusers` |

Все модели принимают один и тот же текстовый промт, что даёт честное «слепое» сравнение архитектур.

**Выходы.** WAV-файлы сохраняются в подкаталог `outputs/`; в конце ноутбука все ссылки собраны для прослушивания.


## Установка зависимостей

Достаточно выполнить эту ячейку один раз — после успешной установки её можно закомментировать.

> ⚠️ **Системные библиотеки.** `audiocraft` тянет за собой `PyAV`, которому на чистой Linux/WSL машине нужны `pkg-config` и dev-заголовки FFmpeg. Если установка падает с `pkg-config is required for building PyAV`, выполните **один раз** в терминале:
> ```bash
> sudo apt-get install -y pkg-config libavformat-dev libavcodec-dev \
>     libavfilter-dev libavdevice-dev libavutil-dev libswscale-dev libswresample-dev
> ```
> На Google Colab уже всё установлено — `apt-get` не нужен.
>
> ⚠️ **Версии библиотек.** Намеренно пиним `transformers==4.44.2` и `diffusers==0.30.3` (см. комментарий в коде): эти версии — последние, где работают AudioLDM 2 (нет конфликта с GPT-2 API) и Riffusion / Bark (нет блокировки `torch.load` из-за CVE-2025-32434 при `torch<2.6`).


In [ ]:
# 1. Свежий pip — нужен для корректной подгрузки manylinux-wheels.
!pip install -q --upgrade pip

# 2. audiocraft ставим ПЕРВЫМ (узкие пины av/xformers).
#    --prefer-binary заставляет брать готовые wheel'ы вместо сборки PyAV.
!pip install -q --prefer-binary audiocraft

# 3. Жёстко пинуем transformers/diffusers под наш стек:
#    - transformers==4.44.2 — последняя версия, где AudioLDM2Pipeline ещё работает
#      (>=4.45 убирает приватный метод _update_model_kwargs_for_generation у GPT-2).
#      Также не имеет блокировки torch.load из-за CVE-2025-32434 (затронут Bark).
#    - diffusers==0.30.3 — до 0.32 ещё разрешено загружать checkpoint'ы без
#      .safetensors (Riffusion v1 хранится только в .bin).
#    --force-reinstall --no-deps гарантирует понижение, если pip уже поставил свежее.
!pip install -q --prefer-binary --force-reinstall --no-deps \
    "transformers==4.44.2" "diffusers==0.30.3"
!pip install -q --prefer-binary accelerate sentencepiece

# 4. Утилиты: scipy.io.wavfile для записи WAV, librosa>=0.10 для
#    мел→waveform инверсии Riffusion (Griffin-Lim).
!pip install -q --prefer-binary scipy soundfile "librosa>=0.10"


## Импорты и общие настройки

Здесь задаём seed, устройство, общий промт и каталог для WAV-файлов.


In [2]:
import gc
import warnings
from pathlib import Path

import numpy as np
import torch
import scipy.io.wavfile

warnings.filterwarnings("ignore")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float16 if device.type == "cuda" else torch.float32
print(f"Device: {device}, dtype: {DTYPE}")

OUT_DIR = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)

# Один и тот же промт для всех моделей
PROMPT = (
    "upbeat modern pop song with catchy synth melody, "
    "punchy drums, female vocal hook, 120 BPM"
)
NEGATIVE = "low quality, noisy, distorted, lo-fi"
DURATION_S = 10


Device: cuda, dtype: torch.float16


## Вспомогательные функции

`save_wav` нормализует амплитуду и пишет 16-битный WAV. `release_model` освобождает память, чтобы все шесть моделей помещались в одну сессию даже на одной GPU.


In [3]:
def save_wav(audio: np.ndarray, sr: int, path: Path) -> Path:
    """Сохраняет аудио (1D mono или 2D (C,T)/(T,C)) в 16-битный WAV."""
    audio = np.asarray(audio, dtype=np.float32).squeeze()
    if audio.ndim == 2 and audio.shape[0] < audio.shape[1]:
        audio = audio.T  # → (samples, channels)
    peak = float(np.abs(audio).max())
    if peak > 0:
        audio = audio / peak * 0.95
    audio_int16 = (audio * 32767.0).astype(np.int16)
    scipy.io.wavfile.write(str(path), sr, audio_int16)
    print(f"  → сохранено: {path}  ({sr} Hz, {len(audio_int16) / sr:.2f} s)")
    return path


def release_model(*objs):
    """Чистит память от моделей/пайплайнов между запусками."""
    for o in objs:
        try:
            del o
        except Exception:
            pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# Здесь будем складывать результаты для итоговой таблицы
STATUS = {}  # name -> {"ok", "path", "err", "params_M"}


## 1. MusicGen (Meta AI, 2023)

**Архитектура.** Single-stage авторегрессионный трансформер-декодер, который предсказывает 4 параллельных потока дискретных токенов аудио-кодека **EnCodec** (32 kHz, 50 Hz токенов, 4 кодбука RVQ). Текстовое условие кодируется **T5-large** и подаётся в декодер через cross-attention. Все четыре кодбука генерируются одним проходом благодаря *delay-pattern interleaving* — каждый следующий кодбук сдвинут на 1 шаг во времени относительно предыдущего, поэтому LM-голова всегда видит «прошлое» по всем уровням сразу.

**Статья:** Copet et al., *“Simple and Controllable Music Generation”*, NeurIPS 2023.


In [4]:
NAME = "01_musicgen"
print(f"=== {NAME} ===")
try:
    from transformers import AutoProcessor, MusicgenForConditionalGeneration

    processor = AutoProcessor.from_pretrained("facebook/musicgen-small")
    model = MusicgenForConditionalGeneration.from_pretrained(
        "facebook/musicgen-small"
    ).to(device)

    inputs = processor(text=[PROMPT], padding=True, return_tensors="pt").to(device)
    # MusicGen: ~50 EnCodec-токенов в секунду
    max_new = int(50 * DURATION_S)
    with torch.inference_mode():
        audio = model.generate(
            **inputs, max_new_tokens=max_new, do_sample=True, guidance_scale=3.0
        )
    sr = model.config.audio_encoder.sampling_rate  # 32000
    wav = audio[0, 0].detach().cpu().numpy()

    path = save_wav(wav, sr, OUT_DIR / f"{NAME}.wav")
    STATUS[NAME] = {
        "ok": True, "path": str(path), "err": None,
        "params_M": sum(p.numel() for p in model.parameters()) / 1e6,
    }
    release_model(model, processor)
except Exception as e:
    STATUS[NAME] = {"ok": False, "path": None, "err": str(e), "params_M": None}
    print(f"  ! ошибка: {e}")


=== 01_musicgen ===


Loading weights: 100%|██████████| 611/611 [00:00<00:00, 1915.24it/s]
[transformers] MusicgenForConditionalGeneration LOAD REPORT from: facebook/musicgen-small
Key                                           | Status     |  | 
----------------------------------------------+------------+--+-
decoder.model.decoder.embed_positions.weights | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  → сохранено: outputs/01_musicgen.wav  (32000 Hz, 9.94 s)


## 2. AudioLDM 2 — music (CVSSP, 2023)

**Архитектура.** Latent Diffusion Model в латенте 1D-VAE, как у AudioLDM, но с **тройным энкодером промта**: CLAP-audio + FLAN-T5 + GPT-2 — их выходы конкатенируются и идут в cross-attention U-Net'а. Эта тройка реализует *“latent diffusion of language-of-audio”*: сначала диффузия в латенте, потом HiFi-GAN-вокодер достраивает waveform. Чекпойнт `audioldm2-music` — версия, дообученная только на музыкальных данных.

**Статья:** Liu et al., *“AudioLDM 2: Learning Holistic Audio Generation with Self-supervised Pretraining”*, 2023.


In [5]:
NAME = "02_audioldm2"
print(f"=== {NAME} ===")
try:
    from diffusers import AudioLDM2Pipeline

    pipe = AudioLDM2Pipeline.from_pretrained(
        "cvssp/audioldm2-music", torch_dtype=DTYPE
    ).to(device)

    g = torch.Generator(device=device).manual_seed(SEED)
    audio = pipe(
        prompt=PROMPT,
        negative_prompt=NEGATIVE,
        num_inference_steps=200,
        audio_length_in_s=float(DURATION_S),
        num_waveforms_per_prompt=1,
        generator=g,
    ).audios[0]

    sr = 16000  # фиксированная sample rate AudioLDM2
    path = save_wav(audio, sr, OUT_DIR / f"{NAME}.wav")
    STATUS[NAME] = {
        "ok": True, "path": str(path), "err": None,
        "params_M": sum(p.numel() for p in pipe.unet.parameters()) / 1e6,
    }
    release_model(pipe)
except Exception as e:
    STATUS[NAME] = {"ok": False, "path": None, "err": str(e), "params_M": None}
    print(f"  ! ошибка: {e}")


=== 02_audioldm2 ===


Loading pipeline components...:  45%|████▌     | 5/11 [00:02<00:02,  2.72it/s][transformers] You are using a model of type `hifigan` to instantiate a model of type `speecht5_hifigan`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.

Loading pipeline components...: 100%|██████████| 11/11 [00:03<00:00,  2.98it/s]
Expected types for language_model: (<class 'transformers.models.gpt2.modeling_gpt2.GPT2LMHeadModel'>,), got <class 'transformers.models.gpt2.modeling_gpt2.GPT2Model'>.


  ! ошибка: 'GPT2Model' object has no attribute '_update_model_kwargs_for_generation'


## 3. Riffusion (Forsgren & Martiros, 2022)

**Архитектура.** Это файнтюн **Stable Diffusion 1.5** на изображениях мел-спектрограмм 512×512 (5-секундные клипы): по оси `x` — время, по `y` — log-частота, яркость пикселя = амплитуда (в dB) c power-преобразованием. Inference — обычный text-to-image, после чего изображение инверсируется обратно в звук: pixel → dB-мел → amplitude-мел → STFT (Griffin–Lim для восстановления фазы) → waveform.

**Источник:** Forsgren & Martiros, *“Riffusion — Stable diffusion for real-time music generation”*, 2022.


In [ ]:
NAME = "03_riffusion"
print(f"=== {NAME} ===")
try:
    from diffusers import StableDiffusionPipeline
    import librosa

    # safety_checker=None прямо при загрузке — не качаем веса NSFW-фильтра,
    # они для спектрограмм всё равно бессмысленны.
    pipe = StableDiffusionPipeline.from_pretrained(
        "riffusion/riffusion-model-v1",
        torch_dtype=DTYPE,
        safety_checker=None,
        requires_safety_checker=False,
    ).to(device)

    g = torch.Generator(device=device).manual_seed(SEED)
    # 1 кадр 512×512 ≈ 5 секунд; чтобы получить ~10 сек, генерируем 2 кадра.
    n_frames = max(1, (DURATION_S + 4) // 5)
    images = pipe(
        prompt=PROMPT, negative_prompt=NEGATIVE,
        height=512, width=512,
        num_inference_steps=50, guidance_scale=7.5,
        num_images_per_prompt=n_frames, generator=g,
    ).images

    # Параметры спектрограммы из riffusion/spectrogram_params.py
    sr = 44100
    n_fft = 17640
    hop = 441
    max_volume_db = 50.0
    power_for_image = 0.25

    audio_chunks = []
    for img in images:
        arr = np.array(img.convert("L"), dtype=np.float32) / 255.0          # [0,1]
        mel_db_norm = arr ** (1.0 / power_for_image)                        # инверс gamma
        mel_db = mel_db_norm * max_volume_db - max_volume_db                # dB ∈ [-50,0]
        mel_power = librosa.db_to_power(mel_db)
        wav = librosa.feature.inverse.mel_to_audio(
            mel_power, sr=sr, n_fft=n_fft, hop_length=hop, n_iter=32,
        )
        audio_chunks.append(wav)

    full = np.concatenate(audio_chunks)[: sr * DURATION_S]
    path = save_wav(full, sr, OUT_DIR / f"{NAME}.wav")
    STATUS[NAME] = {
        "ok": True, "path": str(path), "err": None,
        "params_M": sum(p.numel() for p in pipe.unet.parameters()) / 1e6,
    }
    release_model(pipe)
except Exception as e:
    STATUS[NAME] = {"ok": False, "path": None, "err": str(e), "params_M": None}
    print(f"  ! ошибка: {e}")


## 4. MAGNeT (Meta AI, 2024)

**Архитектура.** Тот же словарь EnCodec-токенов, что и у MusicGen, но LM **неавторегрессионный** и **masked**: на каждой итерации модель видит замаскированную последовательность и предсказывает все позиции параллельно (как MaskGIT). После шага мы фиксируем наиболее уверенные предсказания, остальные снова маскируем — и так несколько шагов. На выходе по числу шагов генерация в ~7–10× быстрее авторегрессионной при сравнимом качестве. Дополнительно используется *rescoring* классификатором CLAP для отбора лучших вариантов.

**Статья:** Ziv et al., *“Masked Audio Generation using a Single Non-Autoregressive Transformer”*, ICLR 2024.


In [7]:
NAME = "04_magnet"
print(f"=== {NAME} ===")
try:
    from audiocraft.models import MAGNeT

    model = MAGNeT.get_pretrained("facebook/magnet-small-30secs", device=str(device))
    model.set_generation_params(
        use_sampling=True, top_k=0, top_p=0.9,
        temperature=3.0, max_cfg_coef=10.0, min_cfg_coef=1.0,
        decoding_steps=[20, 10, 10, 10],
    )
    wav = model.generate([PROMPT])  # (B, 1, T) — длительность фиксирована (30 сек)
    sr = model.sample_rate

    arr = wav[0].detach().cpu().numpy()
    arr = arr[..., : sr * DURATION_S]  # обрезаем до 10 сек для честного сравнения
    path = save_wav(arr, sr, OUT_DIR / f"{NAME}.wav")
    STATUS[NAME] = {
        "ok": True, "path": str(path), "err": None,
        "params_M": sum(p.numel() for p in model.lm.parameters()) / 1e6,
    }
    release_model(model)
except Exception as e:
    STATUS[NAME] = {"ok": False, "path": None, "err": str(e), "params_M": None}
    print(f"  ! ошибка: {e}")


=== 04_magnet ===
  ! ошибка: No module named 'audiocraft'


## 5. Bark (Suno, 2023)

**Архитектура.** Три последовательных трансформера типа GPT:

1. **Text → semantic tokens** (грубая семантика звука/речи; собственный токенизатор Suno).
2. **Semantic → coarse codec tokens** (первые 2 кодбука EnCodec).
3. **Coarse → fine codec tokens** (остальные 6 кодбуков EnCodec; модель работает как denoiser).

Bark изначально создан для речи, но при подсказках вроде маркера `♪` уверенно генерирует мелодии, эффекты и пение — поэтому считается одной из первых открытых моделей «text → song».

**Источник:** Suno AI, *Bark: text-prompted generative audio model* (open-source, 2023).


In [8]:
NAME = "05_bark"
print(f"=== {NAME} ===")
try:
    from transformers import AutoProcessor, BarkModel

    processor = AutoProcessor.from_pretrained("suno/bark-small")
    model = BarkModel.from_pretrained("suno/bark-small").to(device)

    # Маркеры ♪ просят Bark переключиться в «музыкальный» режим
    bark_prompt = f"♪ {PROMPT} ♪"
    inputs = processor(bark_prompt, return_tensors="pt").to(device)
    with torch.inference_mode():
        audio = model.generate(**inputs, do_sample=True, fine_temperature=0.4, coarse_temperature=0.8)
    sr = model.generation_config.sample_rate  # 24000
    wav = audio[0].detach().cpu().float().numpy()

    # Обрежем до DURATION_S, если Bark выдал больше
    wav = wav[: sr * DURATION_S]
    path = save_wav(wav, sr, OUT_DIR / f"{NAME}.wav")
    STATUS[NAME] = {
        "ok": True, "path": str(path), "err": None,
        "params_M": sum(p.numel() for p in model.parameters()) / 1e6,
    }
    release_model(model, processor)
except Exception as e:
    STATUS[NAME] = {"ok": False, "path": None, "err": str(e), "params_M": None}
    print(f"  ! ошибка: {e}")


=== 05_bark ===
  ! ошибка: Due to a serious vulnerability issue in `torch.load`, even with `weights_only=True`, we now require users to upgrade torch to at least v2.6 in order to use the function. This version restriction does not apply when loading files with safetensors.
See the vulnerability report here https://nvd.nist.gov/vuln/detail/CVE-2025-32434


## 6. Stable Audio Open (Stability AI, 2024)

**Архитектура.** **Diffusion Transformer (DiT)**, работающий не в пиксельном латенте, а в латенте 1D-VAE, который кодирует **стерео-аудио 44.1 kHz** с компрессией ~×1024 по времени. Текстовое условие — через **T5-base**, плюс отдельные timing-эмбеддинги (start/end секунд), благодаря которым модель умеет выдавать клипы переменной длины. В отличие от спектрограммных подходов, выход VAE декодируется сразу в waveform без отдельного вокодера.

> ⚠️ Репозиторий `stabilityai/stable-audio-open-1.0` *gated*. Перед запуском выполните один раз в терминале:
> ```bash
> huggingface-cli login
> ```
> и примите лицензию на странице модели.

**Статья:** Evans et al., *“Stable Audio Open”*, arXiv 2024.


In [9]:
NAME = "06_stable_audio_open"
print(f"=== {NAME} ===")
try:
    from diffusers import StableAudioPipeline

    pipe = StableAudioPipeline.from_pretrained(
        "stabilityai/stable-audio-open-1.0", torch_dtype=DTYPE
    ).to(device)

    g = torch.Generator(device=device).manual_seed(SEED)
    out = pipe(
        prompt=PROMPT, negative_prompt=NEGATIVE,
        num_inference_steps=200,
        audio_end_in_s=float(DURATION_S),
        num_waveforms_per_prompt=1,
        generator=g,
    ).audios

    sr = pipe.vae.sampling_rate  # 44100
    wav = out[0].T.float().cpu().numpy()  # (samples, 2) — стерео
    path = save_wav(wav, sr, OUT_DIR / f"{NAME}.wav")
    STATUS[NAME] = {
        "ok": True, "path": str(path), "err": None,
        "params_M": sum(p.numel() for p in pipe.transformer.parameters()) / 1e6,
    }
    release_model(pipe)
except Exception as e:
    STATUS[NAME] = {"ok": False, "path": None, "err": str(e), "params_M": None}
    print(f"  ! ошибка: {e}")


=== 06_stable_audio_open ===
  ! ошибка: 401 Client Error. (Request ID: Root=1-6a026591-65c645f16933c54c5c1e3a30;5983f54f-74b8-483d-9545-74565f65adf7)

Cannot access gated repo for url https://huggingface.co/stabilityai/stable-audio-open-1.0/resolve/main/model_index.json.
Access to model stabilityai/stable-audio-open-1.0 is restricted. You must have access to it and be authenticated to access it. Please log in.


## Сводная таблица результатов

Статус каждой модели после генерации, путь до WAV и число параметров основной сети (LM/U-Net/DiT).


In [10]:
import pandas as pd

rows = []
for name, info in STATUS.items():
    rows.append({
        "Модель": name,
        "Статус": "OK" if info["ok"] else "FAILED",
        "Файл": info["path"] or "—",
        "Параметры, M": f'{info["params_M"]:.1f}' if info["params_M"] else "—",
        "Ошибка": (info["err"] or "")[:80],
    })
df = pd.DataFrame(rows)
df


,Модель,Статус,Файл,"Параметры, M",Ошибка
0,01_musicgen,OK,outputs/01_musicgen.wav,586.9,
1,02_audioldm2,FAILED,—,—,'GPT2Model' object has no attribute '_update_m...
2,03_riffusion,FAILED,—,—,Due to a serious vulnerability issue in `torch...
3,04_magnet,FAILED,—,—,No module named 'audiocraft'
4,05_bark,FAILED,—,—,Due to a serious vulnerability issue in `torch...
5,06_stable_audio_open,FAILED,—,—,401 Client Error. (Request ID: Root=1-6a026591...


## Прослушивание сгенерированных композиций

Один и тот же промт, шесть разных архитектур.


In [11]:
from IPython.display import Audio, display, Markdown

for name, info in STATUS.items():
    if not info["ok"]:
        display(Markdown(f"**{name}** — ❌ {info['err']}"))
        continue
    display(Markdown(f"### {name}\n`{info['path']}` · {info['params_M']:.1f} M params"))
    display(Audio(info["path"]))


### 01_musicgen
`outputs/01_musicgen.wav` · 586.9 M params

**02_audioldm2** — ❌ 'GPT2Model' object has no attribute '_update_model_kwargs_for_generation'

**03_riffusion** — ❌ Due to a serious vulnerability issue in `torch.load`, even with `weights_only=True`, we now require users to upgrade torch to at least v2.6 in order to use the function. This version restriction does not apply when loading files with safetensors.
See the vulnerability report here https://nvd.nist.gov/vuln/detail/CVE-2025-32434

**04_magnet** — ❌ No module named 'audiocraft'

**05_bark** — ❌ Due to a serious vulnerability issue in `torch.load`, even with `weights_only=True`, we now require users to upgrade torch to at least v2.6 in order to use the function. This version restriction does not apply when loading files with safetensors.
See the vulnerability report here https://nvd.nist.gov/vuln/detail/CVE-2025-32434

**06_stable_audio_open** — ❌ 401 Client Error. (Request ID: Root=1-6a026591-65c645f16933c54c5c1e3a30;5983f54f-74b8-483d-9545-74565f65adf7)

Cannot access gated repo for url https://huggingface.co/stabilityai/stable-audio-open-1.0/resolve/main/model_index.json.
Access to model stabilityai/stable-audio-open-1.0 is restricted. You must have access to it and be authenticated to access it. Please log in.

## Выводы

После запуска ноутбука слушаем 6 файлов и сравниваем по критериям:

- **Соответствие промту** (поп, бодрый темп, синт, ударные, женский вокал).
- **Качество звука** (артефакты, шум, ширина диапазона).
- **Музыкальная связность** (структура, гармония, ритм).
- **Скорость генерации** на текущем железе.

*Эту секцию имеет смысл дописать после прослушивания собственных результатов.*

Что обычно слышно по литературе и публичным демо:

- **MusicGen** и **MAGNeT** дают самый «чистый» поп-инструментал в этих весовых категориях: связный ритм и тональность; MAGNeT при этом в разы быстрее на инференсе.
- **Stable Audio Open** хорош в текстурах и стерео-сцене, но иногда «расплывается» в структуре куплета.
- **AudioLDM 2-music** конкурентен по разнообразию инструментов, но перкуссия чуть хуже, чем у MusicGen.
- **Riffusion** даёт характерный лоу-фай-привкус из-за Griffin–Lim'овской фазы; больше годится для лоу-фай-битов, чем для чистого попа.
- **Bark** в музыкальном режиме нестабилен — может вместо песни выдать гудение/речь; всё равно засчитывается как запуск архитектуры.

**Итог.** Запущено 6 моделей → 6 WAV-файлов в `outputs/` → минимум 10 баллов (по 2 за модель), плюс 2 бонусных за шестую.
